## 10 — Temporal features & GEE models: pressure → shot probability

Extends the features from notebook 09 with shot and xG history, writes the result as `gold.match_momentum_temporal`, then fits a sequence of GEE (Generalized Estimating Equations) models with binary outcome `target_shot`.

Models:
- **M0** — baseline: match minute + team + location (home/away)
- **M1** — M0 + shot in previous minute + shots in 5 min + xG in 5 min
- **M2** — M1 + own pressure + opponent pressure + pressure change
- **M3** — M1 + pressure advantage + pressure change

Grouping structure: `team_match_id` = `sportmonks_fixture_id_sportmonks_team_id`. Within-group correlation modelled as AR(1).


### 1. Load features table 

Reads `gold.match_momentum_features` from notebook 09.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

features_df = spark.table(
    "wsl_analytics.gold.match_momentum_features"
)

display(features_df)

### 2. Define base window 

Temporal window partitioned by (`sportmonks_fixture_id`, `sportmonks_team_id`) and ordered by `minute` — base for all lag and rolling window operations.


In [ ]:
team_match_window = (
    Window
    .partitionBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id"
    )
    .orderBy("minute")
)

### 3. Define rolling windows 

`prev_3m` and `prev_5m` windows with `rowsBetween(-3/-5, -1)` — exclude the current minute.


In [ ]:
prev_3m = team_match_window.rowsBetween(-3, -1)
prev_5m = team_match_window.rowsBetween(-5, -1)

### 4. Create team_match_id 

Concatenates `sportmonks_fixture_id` and `sportmonks_team_id` into a single string key `team_match_id`. This will be the GEE grouping variable (one time series = one `team_match_id`).


In [ ]:
temporal_df = (
    features_df
    .withColumn(
        "team_match_id",
        F.concat_ws(
            "_",
            F.col("sportmonks_fixture_id"),
            F.col("sportmonks_team_id")
        )
    )
)

### 5. shot_lag_1 

`F.lag("target_shot", 1)` — binary indicator for a shot in the previous minute (t-1). Strong predictor of shot clustering (attacks are rarely isolated).


In [ ]:
temporal_df = (
    temporal_df
    .withColumn(
        "shot_lag_1",
        F.lag(
            "target_shot",
            1
        ).over(team_match_window)
    )
)

### 6. Shots in rolling windows 

`shots_prev_3m_count` and `shots_prev_5m_count` — total shot counts in the previous 3 and 5 minutes. Measure offensive momentum.


In [ ]:
temporal_df = (
    temporal_df

    .withColumn(
        "shots_prev_3m_count",
        F.sum(
            "target_shot_count"
        ).over(prev_3m)
    )

    .withColumn(
        "shots_prev_5m_count",
        F.sum(
            "target_shot_count"
        ).over(prev_5m)
    )
)

### 7. xG in rolling window 

`xg_prev_5m` — total xG in the previous 5 minutes. Measures not just the occurrence of shots but their quality.


In [ ]:
temporal_df = (
    temporal_df
    .withColumn(
        "xg_prev_5m",
        F.sum(
            "target_xg"
        ).over(prev_5m)
    )
)

### 8. Shots on target in window 

`shots_on_target_prev_5m` — number of on-target shots in the previous 5 minutes.


In [ ]:
temporal_df = (
    temporal_df
    .withColumn(
        "shots_on_target_prev_5m",
        F.sum(
            "target_shot_on_target"
        ).over(prev_5m)
    )
)

### 9. had_shot_prev_5m flag 

Binary version of `shots_prev_5m_count > 0` — whether the team took at least one shot in the previous 5 minutes.


In [ ]:
temporal_df = (
    temporal_df
    .withColumn(
        "had_shot_prev_5m",
        (
            F.col("shots_prev_5m_count") > 0
        ).cast("int")
    )
)

### 10. temporal_prev_5m_n 

Number of rows in the 5-minute window (1–5). Used for filtering: GEE models are fitted only on records with `temporal_prev_5m_n == 5` to ensure a complete shot history.


In [ ]:
temporal_df = (
    temporal_df
    .withColumn(
        "temporal_prev_5m_n",
        F.count(
            F.lit(1)
        ).over(prev_5m)
    )
)

### 11. Preview sample fixture 

Displays temporal columns for one fixture — window correctness check.


In [ ]:
example_fixture = (
    temporal_df
    .select("sportmonks_fixture_id")
    .first()[0]
)

display(
    temporal_df
    .filter(
        F.col("sportmonks_fixture_id")
        == example_fixture
    )
    .select(
        "sportmonks_fixture_id",
        "sportmonks_team_name",
        "minute",
        "target_shot",
        "shot_lag_1",
        "shots_prev_3m_count",
        "shots_prev_5m_count",
        "xg_prev_5m",
        "pressure_prev_5m_mean",
        "pressure_change_5m"
    )
    .orderBy(
        "sportmonks_team_id",
        "minute"
    )
)

### 12. Write gold.match_momentum_temporal 

Writes the Gold table with temporal features.


In [ ]:
(
    temporal_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.gold.match_momentum_temporal"
    )
)

### 13. Filter for modeling 

Filters data for GEE modeling: complete temporal window (5 minutes), complete own pressure window (5 minutes), complete opponent pressure window (5 minutes), and non-NULL target.


In [ ]:
modeling_spark_df = (
    temporal_df

    .filter(
        F.col("temporal_prev_5m_n") == 5
    )

    .filter(
        F.col("pressure_prev_5m_n") == 5
    )

    .filter(
        F.col("opponent_pressure_prev_5m_n") == 5
    )

    .filter(
        F.col("target_shot").isNotNull()
    )

    .select(
        "team_match_id",

        "sportmonks_fixture_id",
        "sportmonks_team_id",

        "location",
        "minute",

        "target_shot",

        "shot_lag_1",
        "shots_prev_3m_count",
        "shots_prev_5m_count",
        "xg_prev_5m",

        "pressure_prev_5m_mean",
        "pressure_prev_5m_max",
        "pressure_change_5m",

        "opponent_pressure_prev_5m_mean",
        "pressure_advantage_prev_5m"
    )
)

### 14. Count observations 

Prints the number of observations and unique `team_match_id` values after filtering.


In [ ]:
print(
    "N observations:",
    modeling_spark_df.count()
)

print(
    "N team-match:",
    modeling_spark_df
    .select("team_match_id")
    .distinct()
    .count()
)

### 15. Convert to pandas

Converts to pandas and sorts by `(team_match_id, minute)` — important for correct AR(1) in GEE.


In [ ]:
model_df = (
    modeling_spark_df
    .toPandas()
    .sort_values(
        [
            "team_match_id",
            "minute"
        ]
    )
    .reset_index(drop=True)
)

### 16. Preview 

Preview of first rows.


In [ ]:
model_df.head()

### 17. Define pressure variables 

List of pressure variables to standardise.


In [ ]:
pressure_variables = [
    "pressure_prev_5m_mean",
    "pressure_prev_5m_max",
    "pressure_change_5m",
    "opponent_pressure_prev_5m_mean",
    "pressure_advantage_prev_5m"
]

### 18. Standardise pressure 

Standardises pressure variables to z-score scale (mean=0, sd=1). Enables comparison of effect sizes across variables and aids GEE model convergence.


In [ ]:
for var in pressure_variables:

    mean = model_df[var].mean()
    sd = model_df[var].std()

    model_df[f"{var}_z"] = (
        model_df[var] - mean
    ) / sd

### 19. Check standardised variables 

Descriptive statistics of standardised variables — verifies mean≈0 and sd≈1.


In [ ]:
model_df[
    [
        "pressure_prev_5m_mean_z",
        "pressure_change_5m_z",
        "pressure_advantage_prev_5m_z"
    ]
].describe()

### 20. Install statsmodels 

Installs statsmodels if not available in the cluster.


In [ ]:
%pip install statsmodels

### 21. Import statsmodels 

Imports required modules.


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

### 22. GEE setup 

Defines family (`Binomial` for binary outcome) and correlation structure (`Autoregressive` with `grid=True` — assumes AR(1) evenly spaced in time, matching the minute grid).


In [ ]:
family = sm.families.Binomial()

ar1 = sm.cov_struct.Autoregressive(
    grid=True
)

### 23. M0 formula 

Baseline model: quadratic minute function (match time is non-linear) + location effect (home/away) + team fixed effects.


In [ ]:
formula_0 = """
target_shot ~
    minute
    + I(minute ** 2)
    + C(location)
    + C(sportmonks_team_id)
"""

### 24. Fit M0 

Defines the GEE model M0.


In [ ]:
model_0 = smf.gee(
    formula=formula_0,

    groups="team_match_id",

    time="minute",

    data=model_df,

    family=family,

    cov_struct=sm.cov_struct.Autoregressive(
        grid=True
    )
)

### 25. M0 results 

`maxiter=300` — iteration limit for the IRLS algorithm. `cov_type="robust"` — robust (sandwich) covariance matrix, does not assume correct correlation structure specification.


In [ ]:
result_0 = model_0.fit(
    maxiter=300,
    cov_type="robust"
)

print("Converged:", result_0.converged)
print("AR(1) rho:", result_0.cov_struct.dep_params)

print(result_0.summary())

### 26. M1 formula 

M1 adds shot history variables: `shot_lag_1` (shot at t-1), `shots_prev_5m_count`, and `xg_prev_5m`.


In [ ]:
formula_1 = """
target_shot ~
    minute
    + I(minute ** 2)
    + C(location)
    + C(sportmonks_team_id)

    + shot_lag_1
    + shots_prev_5m_count
    + xg_prev_5m
"""

### 27. Fit M1 

Fits and prints M1 results.


In [ ]:
model_1 = smf.gee(
    formula=formula_1,

    groups="team_match_id",

    time="minute",

    data=model_df,

    family=family,

    cov_struct=sm.cov_struct.Autoregressive(
        grid=True
    )
)

result_1 = model_1.fit(
    maxiter=300,
    cov_type="robust"
)

print("Converged:", result_1.converged)
print("AR(1) rho:", result_1.cov_struct.dep_params)

print(result_1.summary())

### 28. M2 formula — pressure 

M2 adds three pressure variables (standardised): own 5-min pressure, opponent 5-min pressure, and 5-min pressure change.


In [ ]:
formula_2 = """
target_shot ~
    minute
    + I(minute ** 2)
    + C(location)
    + C(sportmonks_team_id)

    + shot_lag_1
    + shots_prev_5m_count
    + xg_prev_5m

    + pressure_prev_5m_mean_z
    + opponent_pressure_prev_5m_mean_z
    + pressure_change_5m_z
"""

### 29. Fit M2 

Fits M2 and prints convergence, AR(1) parameter, and summary.


In [ ]:
model_2 = smf.gee(
    formula=formula_2,

    groups="team_match_id",

    time="minute",

    data=model_df,

    family=family,

    cov_struct=sm.cov_struct.Autoregressive(
        grid=True
    )
)

result_2 = model_2.fit(
    maxiter=300,
    cov_type="robust"
)

print("Converged:", result_2.converged)
print("AR(1) rho:", result_2.cov_struct.dep_params)

print(result_2.summary())

### 30. M3 formula — pressure advantage 

M3 replaces separate own and opponent pressure variables with a single **pressure advantage** variable (difference) and pressure change. This is a more parsimonious specification.


In [ ]:
formula_3 = """
target_shot ~
    minute
    + I(minute ** 2)
    + C(location)
    + C(sportmonks_team_id)

    + shot_lag_1
    + shots_prev_5m_count
    + xg_prev_5m

    + pressure_advantage_prev_5m_z
    + pressure_change_5m_z
"""

### 31. Fit M3 

Fits M3.


In [ ]:
model_3 = smf.gee(
    formula=formula_3,

    groups="team_match_id",

    time="minute",

    data=model_df,

    family=family,

    cov_struct=sm.cov_struct.Autoregressive(
        grid=True
    )
)

result_3 = model_3.fit(
    maxiter=300,
    cov_type="robust"
)

print("Converged:", result_3.converged)
print("AR(1) rho:", result_3.cov_struct.dep_params)

print(result_3.summary())

### 32. Odds ratios helper 

`gee_odds_ratios()` computes odds ratios (exp(B)), confidence intervals, and p-values from GEE results.


In [ ]:
import numpy as np
import pandas as pd

def gee_odds_ratios(result):

    ci = result.conf_int()

    table = pd.DataFrame({
        "B": result.params,
        "SE": result.bse,
        "p": result.pvalues,

        "OR": np.exp(
            result.params
        ),

        "OR_CI_low": np.exp(
            ci[0]
        ),

        "OR_CI_high": np.exp(
            ci[1]
        )
    })

    return table

### 33. OR table for M2 

Odds ratio table for M2 — prints the full table.


In [ ]:
or_model_2 = gee_odds_ratios(
    result_2
)

display(or_model_2)

### 34. OR for pressure variables 

Prints ORs only for the pressure variables from M2 — this is the main analytical result.


In [ ]:
or_model_2.loc[
    [
        "pressure_prev_5m_mean_z",
        "opponent_pressure_prev_5m_mean_z",
        "pressure_change_5m_z"
    ]
]

### 35. AR(1) parameter / Parametr AR(1)

**PL:** AR(1) rho z M2 — mierzy korelację pomiędzy minutami w tej samej serii `team_match_id`.  
**EN:** AR(1) rho from M2 — measures the correlation between adjacent minutes within the same `team_match_id` series.


In [ ]:
print(
    "AR(1) rho:",
    result_2.cov_struct.dep_params
)

### 36. QIC model comparison / Porównanie modeli QIC

**PL:** QIC (Quasi-Information Criterion) to odpowiednik AIC dla modeli GEE. Niższy QIC = lepsze dopasowanie. Porównujemy wszystkie 4 modele.  
**EN:** QIC (Quasi-Information Criterion) is the GEE analogue of AIC. Lower QIC = better fit. Compares all 4 models.


In [ ]:
qic_0 = result_0.qic()
qic_1 = result_1.qic()
qic_2 = result_2.qic()
qic_3 = result_3.qic()

qic_df = pd.DataFrame({
    "model": [
        "M0 baseline",
        "M1 previous attack",
        "M2 pressure",
        "M3 pressure advantage"
    ],

    "QIC": [
        qic_0[0],
        qic_1[0],
        qic_2[0],
        qic_3[0]
    ],

    "QICu": [
        qic_0[1],
        qic_1[1],
        qic_2[1],
        qic_3[1]
    ]
})

qic_df